# PCA on Drilling Data
## Machine learning for energy applications — Module 4, Exercise 1

In this exercise we apply **Principal Component Analysis (PCA)** to a drilling dataset.
PCA is one of the most widely used dimensionality-reduction techniques: it finds
orthogonal directions (principal components) that capture maximum variance in the data.

We will:
1. Explore and standardize the drilling features
2. Fit PCA and examine the explained variance
3. Visualize the data in the reduced PC space
4. Interpret the PCA loadings
5. Reconstruct the data and measure reconstruction error
6. Use PCA as preprocessing for a downstream clustering task

> **Exploratory analysis.** Every fit in this notebook (scaler, PCA, k-means) uses all 100 rows,
> because the question is what structure this one drilling table has. The results describe these
> rows; they are not a model for new data. If PCA fed a predictive model or monitored new depths,
> the scaler and PCA would be fitted on training rows only (for example inside a `Pipeline`), and
> the reconstruction error would be checked on held-out depths.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset contains drilling measurements at various depths:
- **Depth** — measured depth along the wellbore
- **ROP** — rate of penetration
- **RPM** — rotational speed of the drill string
- **Torque** — torque applied to the drill string
- **Weight_on_Bit** — downward force on the drill bit

In [ ]:
# YOUR CODE HERE
# TODO: Load "drilling.csv" into a DataFrame
# TODO: Print the shape
# TODO: Display the first 10 rows


In [ ]:
# YOUR CODE HERE
# TODO: Call df.describe() to see summary statistics


In [ ]:
# YOUR CODE HERE
# TODO: Call df.info() to check data types and missing values


### 1.1 Correlation Matrix Heatmap

PCA is most useful when features are correlated — it rotates the coordinate system
so that the first few components capture most of the variance.

In [ ]:
# YOUR CODE HERE
# TODO: Compute the correlation matrix with df.corr()
# TODO: Plot a heatmap using sns.heatmap with annot=True, cmap="coolwarm"
# Hint: use center=0 and square=True for a clean layout


---
## 2 — Standardize Features

PCA is sensitive to scale — features with larger numeric ranges would dominate the
variance calculation. We standardize to zero mean and unit variance. The scaler is fitted on
all 100 rows because this is an exploratory description of this table.

We treat **Depth** as a positional index rather than a feature to reduce.

In [ ]:
# YOUR CODE HERE
# TODO: Define feature_cols = ["ROP", "RPM", "Torque", "Weight_on_Bit"]
# TODO: Extract the feature values as a numpy array X
# TODO: Use StandardScaler to fit_transform X into X_scaled
# TODO: Print the means and stds after scaling to verify


---
## 3 — Apply PCA (All Components)

We fit PCA retaining **all** components first so we can inspect how variance
is distributed across them.

In [ ]:
# YOUR CODE HERE
# TODO: Fit PCA with n_components equal to the number of features
# TODO: Transform X_scaled
# TODO: Print the explained_variance_ratio_ for each component
# TODO: Print the total explained variance


### 3.1 Scree Plot and Cumulative Variance

In [ ]:
# YOUR CODE HERE
# TODO: Compute cumulative explained variance using np.cumsum()
# TODO: Create a 1x2 subplot figure:
#   Left: bar chart of individual explained variance ratio (scree plot)
#   Right: line plot of cumulative variance with 90% and 95% threshold lines
# Hint: use axhline for the threshold lines


---
## 4 — Choose Number of Components

We select the smallest number of components that capture at least **90-95%** of
the total variance. The threshold is a choice, not a rule: justify it from the purpose (for
example a compression tolerance for each channel), and read the scree plot as well.

In [ ]:
# YOUR CODE HERE
# TODO: Set threshold = 0.95
# TODO: Find n_components using np.argmax(cumulative >= threshold) + 1
# TODO: Print the number of components and the cumulative variance achieved


In [ ]:
# YOUR CODE HERE
# TODO: Fit a new PCA with the chosen n_components
# TODO: Transform X_scaled into X_pca_reduced
# TODO: Print original and reduced shapes. Did the dimension actually go down?

---
## 5 — Visualize Data in PC1 vs PC2 Space

We project the data onto the first two principal components and colour by **Depth**
to see whether there are depth-related patterns in the drilling parameters.

In [ ]:
# YOUR CODE HERE
# TODO: Create a scatter plot of PC1 vs PC2
# TODO: Colour points by df["Depth"] using cmap="viridis"
# TODO: Add a colorbar and label axes with variance percentages
# Hint: use plt.colorbar(scatter, ax=ax)


---
## 6 — Analyse PCA Loadings

The **loadings** (rows of `pca.components_`) tell us how much each original feature
contributes to each principal component. Large absolute loadings indicate strong
contributions.

In [ ]:
# YOUR CODE HERE
# TODO: Create a DataFrame from pca_full.components_.T
#   - columns: ["PC1", "PC2", ...]
#   - index: feature_cols
# TODO: Display the loadings rounded to 4 decimal places


In [ ]:
# YOUR CODE HERE
# TODO: Plot the loadings as a heatmap using sns.heatmap
# Hint: use cmap="RdBu_r", center=0, annot=True


### 6.1 Biplot — Loadings on the PC1-PC2 Plane

A biplot overlays the loading vectors on the scatter plot so we can see which
features drive the spread in PC space.

In [ ]:
# YOUR CODE HERE
# TODO: Scatter the data in PC1-PC2 space (grey, low alpha)
# TODO: For each feature, draw an arrow from origin using pca_full.components_
#   - Scale the arrows by a factor (e.g., 3.0) for visibility
#   - Label each arrow with the feature name
# Hint: use ax.arrow(0, 0, dx, dy, ...) and ax.text()


---
## 7 — Reconstruction from Reduced Dimensions

When we reduce dimensions we lose some information. We can reconstruct an
approximation of the original data using `inverse_transform` and measure how
much error the compression introduces.

Report the error **per feature, in original units** (the CSV does not state units, so use the
file's own). A single error averaged over features with different units would be dominated by
the feature with the largest numbers (Weight_on_Bit). If the chosen number of components equals
the number of features, the reconstruction is exact, so we also compare 1 to 4 components.
These are in-sample errors on the same 100 rows used to fit the scaler and PCA; a compression
decision for new data would check them on held-out depths.

In [ ]:
# YOUR CODE HERE
# TODO: Use pca_reduced.inverse_transform(X_pca_reduced) to get the reconstruction
# TODO: Undo the scaling with scaler.inverse_transform
# TODO: Compute the RMSE of each feature (axis=0) between X and the reconstruction
# TODO: Print the RMSE of each feature in its original units

In [ ]:
# YOUR CODE HERE
# TODO: For each number of components (1 through len(feature_cols)):
#   - Fit PCA, transform, inverse_transform, compute the MSE in standardized units
#   - Undo the scaling and compute the RMSE of each feature in original units
# TODO: Show the per-feature RMSE for every number of components as a table
#       (rows: number of components, columns: features)
# TODO: Plot the standardized reconstruction error vs number of components

---
## 8 — PCA as Preprocessing for Clustering

Dimensionality reduction can help clustering by removing noise dimensions. We compare K-Means
clustering on the standardized features with K-Means on the PCA-reduced data (exploratory: all
fits use the same 100 rows).

Two cautions when reading the result:
- If `n_components` equals the number of features, PCA is only a rotation, so K-Means finds
  the same clusters and the same silhouette score.
- Silhouette scores computed in spaces of different dimension are not directly comparable.
  For a fair comparison, also score the PCA-space clusters in the standardized feature space.

In [ ]:
# YOUR CODE HERE
# TODO: Set n_clusters = 3
# TODO: Fit KMeans on X_scaled and compute silhouette_score
# TODO: Fit KMeans on X_pca_reduced and compute silhouette_score
# TODO: Also score the PCA-space labels in the standardized space: silhouette_score(X_scaled, labels_pca)
# TODO: Print the scores to compare

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure
#   Left: scatter plot of PC1 vs PC2 coloured by original-space cluster labels
#   Right: scatter plot of PC1 vs PC2 coloured by PCA-space cluster labels
# TODO: Include silhouette scores in subplot titles


---
## 9 — Extension (optional): PCA on Raw Units

Repeat PCA without standardizing and compare PC1 with the standardized fit. On raw units the
variance is dominated by the feature with the largest numeric spread, so PC1 then describes the
units of measurement rather than drilling behaviour.

In [ ]:
# YOUR CODE HERE
# TODO: Fit PCA on the raw (unscaled) X and print its explained_variance_ratio_
# TODO: Compare the PC1 loadings of the raw fit and of the standardized fit (pca_full)
#       side by side, together with the standard deviation of each feature

---
## Summary

In this exercise we applied PCA to drilling data and covered:

1. **Data exploration** — correlation structure among drilling parameters.
2. **Standardization** — essential preprocessing for PCA since it is scale-sensitive.
3. **Explained variance** — scree plots and cumulative variance to choose the number of components.
4. **Visualization** — projecting data into PC1-PC2 space and colouring by Depth.
5. **Loadings analysis** — understanding which features drive each principal component (biplot).
6. **Reconstruction error** — per feature, in original units, for each number of components.
7. **PCA + Clustering** — using PCA as a preprocessing step for K-Means, compared in the same space.
8. **Raw units (extension)** — why PC1 of unscaled data follows the largest numeric spread.

**Key takeaway**: PCA is linear, fast, and interpretable, and it compresses well when channels are
correlated. Whether it compresses a particular table is an empirical question: read the spectrum,
the loadings, and the reconstruction error per channel in original units. A flat spectrum means
the channels carry separate information, which is a finding, not a failure. The fits here describe
these 100 rows (exploratory); for prediction or monitoring, fit the scaler and PCA on training
data only.